# ChemiAI — Deep Learning (MLP)


## Что делаем
Обучаем MLP (нейросетевой baseline) на очищенных и масштабированных признаках.
Таргеты переводим в log1p-пространство, затем возвращаемся через expm1.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
TARGET_COLS = ["IC50, mM", "CC50, mM", "SI"]
ID_COL = "index"

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

feature_cols = [c for c in train.columns if c not in [ID_COL, *TARGET_COLS]]
X_train = train[feature_cols].copy()
y_train = train[TARGET_COLS].copy()
X_test = test[feature_cols].copy()


In [ ]:
# Очистка: удаляем константы и точные дубли столбцов
const_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
hashed = {c: pd.util.hash_pandas_object(X_train[c], index=False).sum() for c in X_train.columns}
groups = {}
for c,h in hashed.items():
    groups.setdefault(h, []).append(c)
dup_cols = []
for cols in groups.values():
    if len(cols) <= 1:
        continue
    base = cols[0]
    for c in cols[1:]:
        if X_train[base].equals(X_train[c]):
            dup_cols.append(c)

drop_cols = sorted(set(const_cols + dup_cols))
X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

imp = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train_np = scaler.fit_transform(imp.fit_transform(X_train))
X_test_np = scaler.transform(imp.transform(X_test))
y_log = np.log1p(y_train.values)


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, out_dim)
        )
    def forward(self, x):
        return self.net(x)

def train_mlp(seed=42, epochs=220, lr=1e-3):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = MLP(X_train_np.shape[1])
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    X_tensor = torch.tensor(X_train_np, dtype=torch.float32)
    y_tensor = torch.tensor(y_log, dtype=torch.float32)

    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        pred = model(X_tensor)
        loss = loss_fn(pred, y_tensor)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        pred_test_log = model(torch.tensor(X_test_np, dtype=torch.float32)).numpy()
    return pred_test_log

preds = [train_mlp(seed=s) for s in [42, 2024, 7]]
pred_test_log = np.mean(preds, axis=0)
pred_test = np.expm1(np.clip(pred_test_log, 0, 12))
pred_test = np.clip(pred_test, 0, None)


In [ ]:
submission = sample_submission.copy()
submission["IC50"] = pred_test[:, 0]
submission["CC50"] = pred_test[:, 1]
submission["SI"] = pred_test[:, 2]
submission.to_csv("submission_deep_learning.csv", index=False)
submission.head()
